In [1]:
%load_ext autoreload
%autoreload 2

# Phase 1

In [2]:
import os
import sys
import shutil
import gymnasium as gym
import torch

# Move up 2 levels: Notebooks/training_transformer/ -> Notebooks/ -> Project Root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root set to: {PROJECT_ROOT}")

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

Project root set to: /home/minh-quan/Documents/Haxball project


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1v1 Geometry & Physical Parameters
TEAM_SIZE = 1
PITCH_W = 840.0
PITCH_H = 500.0
GOAL_H = 450.0
ROUND_STEPS = 900
ACTION_REPEAT = 10
NUM_ENVS = 24

SAVE_DIR_S1_P1 = "models/stage1/phase1"
POOL_DIR_S1_P1 = os.path.join(SAVE_DIR_S1_P1, "pool")
os.makedirs(POOL_DIR_S1_P1, exist_ok=True)

Using device: cuda


In [ ]:
def make_s1_p1_env(env_rank: int):
  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S1_P1,
        team="blue",
        device="cpu",
        p_random=1.0,    
        p_heuristic=0.0,  
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H,
        pitch_width=PITCH_W,
        pitch_height=PITCH_H,
        opponent_controller=opp_ctrl,
    )
    env.reset(seed=1000 + env_rank)
    return env
  return _thunk

envs_s1_p1 = gym.vector.AsyncVectorEnv(
    [make_s1_p1_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s1 = TransformerActorCritic().to(device)

# Seed initial self-play pool with randomly initialized weights
init_weights_path = os.path.join(POOL_DIR_S1_P1, "champion.pt")
torch.save(model_s1.state_dict(), init_weights_path)
torch.save(model_s1.state_dict(), os.path.join(POOL_DIR_S1_P1, "history_0.pt"))

print("🚀 Starting Stage 1 - Phase 1: Bootstrapping motor skills...")

train_mappo(
    envs=envs_s1_p1,
    model=model_s1,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=3_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=1e-3,
    lr_final=1e-5,
    ent_coef_init=0.020,
    ent_coef_final=0.001,
    gamma=0.99,
    gae_lambda=0.95,
    active_tiers=["random"],
    target_tier="random",
    #filter_thresholds={"random": 0.50},  # Must defeat random bots >50% to qualify
    #tier_ratios={"random": 0.30, "champion": 0.70},
    eval_episodes=50,
    eval_freq=200_000,
    save_dir=SAVE_DIR_S1_P1,
    pool_dir=POOL_DIR_S1_P1,
    goal_height=GOAL_H,
    pitch_width=PITCH_W,
    pitch_height=PITCH_H,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s1_p1.close()

# Phase 2

In [7]:
SAVE_DIR_S1_P2 = "models/stage1/phase2"
POOL_DIR_S1_P2 = os.path.join(SAVE_DIR_S1_P2, "pool")

PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0  # Regulation net
ROUND_STEPS = 1800
os.makedirs(POOL_DIR_S1_P2, exist_ok=True)

phase1_best = os.path.join(SAVE_DIR_S1_P1, "best_model.pt")
phase1_final = os.path.join(SAVE_DIR_S1_P1, "final_model.pt")
seed_file = phase1_best if os.path.exists(phase1_best) else phase1_final

shutil.copy(seed_file, os.path.join(POOL_DIR_S1_P2, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S1_P2, "history_0.pt"))

print(f"🔥 Phase 2 seeded with weights from: {seed_file}")

🔥 Phase 2 seeded with weights from: models/stage1/phase1/best_model.pt


In [ ]:
def make_s1_p2_env(env_rank: int):
  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S1_P2,
        team="blue",
        device="cpu",
        p_random=0.05,     # 5% Random bot
        p_heuristic=0.35,  # 35% Heuristic bot -> 60% Self-Play
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),
        ],
    )
    env.reset(seed=2000 + env_rank)
    return env
  return _thunk

envs_s1_p2 = gym.vector.AsyncVectorEnv(
    [make_s1_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s1_p2 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model_s1_p2.load_state_dict(state_dict, strict=True)
print("✅ Phase 2 model weights loaded successfully.")

train_mappo(
    envs=envs_s1_p2,
    model=model_s1_p2,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=15_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=5e-5,
    lr_final=3e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.002,
    gamma=0.995,
    gae_lambda=0.95,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",
    filter_thresholds={"heuristic": 0.90},  # Gatekeeper ensures no regression against heuristics
    tier_ratios={"heuristic": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=200_000,
    save_dir=SAVE_DIR_S1_P2,
    pool_dir=POOL_DIR_S1_P2,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s1_p2.close()

✅ Phase 2 model weights loaded successfully.
🚀 Entity-Transformer MAPPO Initialized | Format: 1v1 | Envs: 24 | Batch: 6144 | Device: cuda

📊 [EVALUATION @ Step 202,752 | Rollout SPS: 2081 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  88.0% | Reward: +2.228 | Goals: 65 Scored, 1 Conceded (+64 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: None, Reward: None, Net: None] (Eval took 4.7s)

📊 [EVALUATION @ Step 405,504 | Rollout SPS: 2199 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  72.0% | Reward: +1.868 | Goals: 63 Scored, 4 Conceded (+59 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: None, Reward: None, Net: None] (Eval took 4.5s)

📊 [EVALUATION @ Step 602,112 | Rollout SPS: 2223 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  88.0% | Reward: +2.548 | Goals: 84 Scored, 5 Conceded (+79 Net)
   ❌ Retaining current baseline. Did not pass criteria for c